# Attitude agility — roll

A worked example of the `quicksat` agility budget about the **roll** axis: what the reaction wheels allow, how long a rest-to-rest slew takes, and whether it fits the time an operation gives it. Both the nominal four-wheel case and the one-wheel-failed case are carried through.

Pitch runs the identical machinery and will get its own notebook.

For *why* each step works the way it does — the two inertia case shapes, where the wheel projection comes from, why one failed wheel halves the axis — see `docs/agility_ref.ipynb`. For the orbit the ground track comes from, see `docs/orbit_ref.ipynb`.

In [10]:
# Change log. LAST_CHANGE and CHANGE_NOTE are typed in by hand: update them whenever
# the inputs move -- a wheel changed, an inertia case remeasured, a settling time
# renegotiated -- so that the figures below can be read against what produced them.
# The run time is recorded automatically, and says how stale the outputs stored in
# this notebook are relative to that last change.
from datetime import datetime

LAST_CHANGE = "2026-09-21"
CHANGE_NOTE = "First roll agility budget: three inertia cases, 4 wheels at 26.5 deg."

print(f"last change  {LAST_CHANGE}")
print(f"             {CHANGE_NOTE}")
print(f"last run     {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}")

last change  2026-09-21
             First roll agility budget: three inertia cases, 4 wheels at 26.5 deg.
last run     2026-09-23 12:02 CEST


In [11]:
import os
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from sample/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import u
from quicksat.agility.budget import AgilityBudget, Axis
from quicksat.mass.budget import MassBudget
from quicksat.utils.orbit import Orbit

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## The inputs

Three files. The agility config holds the inertia cases, the wheels and the settling time; the shared orbit gives the ground track speed; the mass budget weighs the spacecraft. The orbit is loaded once and handed over rather than re-read, as the other budgets do it.

The agility config carries **no mass**. The mass budget owns that, and a second copy here would drift — the same reason altitude lives only in `orbit.yaml`. It carries no target duration either: how long a manoeuvre may take is asked of a spacecraft, not a property of one, so it arrives at the call.

In [12]:
DATA = Path("sample") / "data"
orbit = Orbit.from_yaml_file(DATA / "orbit.yaml")
mass_data = MassBudget.from_csv(DATA / "equipment.csv", DATA / "budget_config.yaml")

roll = AgilityBudget.from_yaml_file(
    DATA / "agility_config.yaml",
    DATA / "orbit.yaml",
    Axis.ROLL,
    "first_guess",
    mass_budget=mass_data,
)

config = roll.config
print(f"orbit         {orbit.altitude:~.0f}, ground track {orbit.ground_track_speed:~.3f}")
print(f"wheels        {config.wheels.count} at {config.wheels.elevation:~} elevation")
print(f"settling      {config.settling_time:~}")
print(f"inertia cases {', '.join(config.inertia_cases)}")
print()
print(f"flying        '{roll.case_name}' -- envelope, so the mass comes from the")
print(f"              mass budget at {roll.case.propellant:~.0f} propellant: {roll.mass:~.2f}")
print(f"inertia       {roll.inertia:~.1f}")

orbit         500 km, ground track 7.059 km / s
wheels        4 at 26.5 deg elevation
settling      20 s
inertia cases first_guess, measured_bol, measured_eol

flying        'first_guess' -- envelope, so the mass comes from the
              mass budget at 100 % propellant: 470.62 kg
inertia       262.3 kg * m ** 2


## What the wheels can put about the axis

The pyramid's symmetry axis is along yaw, so roll and pitch both lie in its base plane and see the same fraction of each wheel. That fraction is the projection: `cos(elevation)` tilts the wheel into the plane, `cos(45°)` resolves it onto the axis within it.

Only part of each wheel is available. The momentum use factor and the torque derating are policy, not hardware — what is held back is disturbance storage and control authority during the slew.

In [13]:
print(f"usable per wheel  {roll.usable_momentum_per_wheel:~.3f}"
      f"  and {roll.usable_torque_per_wheel:~.3f}")
print(f"projection        {roll.projection.magnitude:.4f}")
print()
print(f"about roll        {roll.axis_momentum():~.3f}  {roll.axis_torque():~.4f}")
print(f"about yaw         {roll.yaw_momentum:~.2f}   -- the weak axis under this")
print("                                    mounting; no slew case here, but it is")
print("                                    what a yaw manoeuvre lives within")

usable per wheel  1.332 m * N * s  and 0.150 m * N
projection        0.6328

about roll        3.372 m * N * s  0.3797 m * N
about yaw         2.38 m * N * s   -- the weak axis under this
                                    mounting; no slew case here, but it is
                                    what a yaw manoeuvre lives within


## The two limits, and where they swap

Momentum sets the fastest the spacecraft can turn; torque sets how quickly it gets there. Below the crossover angle the wheels never saturate and the slew is **torque limited** — a triangular accelerate-then-decelerate profile. Above it they do, and the slew coasts at maximum rate: **momentum limited**, a trapezoid.

Which one binds is the useful output. Torque limited means more torque would buy something; momentum limited means it would not.

In [14]:
print(f"max rate          {roll.max_rate():~.4f}")
print(f"max acceleration  {roll.max_acceleration():~.5f}")
print(f"crossover angle   {roll.crossover_angle():~.2f}")
print()
for angle in (5 * u.deg, 90 * u.deg):
    print(f"  {angle:~4.0f} is {roll.profile(angle).value:11s} limited by "
          f"{'torque' if angle <= roll.crossover_angle() else 'momentum'}")

max rate          0.7366 deg / s
max acceleration  0.08295 deg / s ** 2
crossover angle   6.54 deg

     5 deg is triangular  limited by torque
    90 deg is trapezoidal limited by momentum


## How long a slew takes

One row per angle: the slew itself, the slew plus settling, the peak rate reached, and how much of the wheel momentum it called on. Momentum used hits 100% for every slew past the crossover — that is what momentum limited means — and the shortfall below it is headroom a larger slew would spend.

In [15]:
roll.slew_table()

,angle,profile,slew_time,total_time,peak_rate,momentum_used
0,5,triangular,15.53,35.53,0.64,0.87
1,10,trapezoidal,22.46,42.46,0.74,1.00
2,15,trapezoidal,29.24,49.24,0.74,1.00
3,20,trapezoidal,36.03,56.03,0.74,1.00
4,40,trapezoidal,63.19,83.19,0.74,1.00
5,45,trapezoidal,69.97,89.97,0.74,1.00
6,60,trapezoidal,90.34,110.34,0.74,1.00
7,90,trapezoidal,131.07,151.07,0.74,1.00


## Does it fit?

The target duration is an argument, because the same spacecraft is asked the question differently by different operations. Give one and the budget says how much room is left; ask it the other way round and it says how far it could have turned instead.

A slew costs swath. Tying the time to the shared orbit's ground track speed turns it into kilometres of ground given up, which is the number a payload operator actually feels.

In [16]:
angle = 40 * u.deg
for target in (113 * u.s, 90 * u.s):
    margin = roll.time_margin(angle, target)
    verdict = "fits" if margin.magnitude >= 0 else "DOES NOT FIT"
    print(f"{angle:~.0f} in {target:~.0f}:  needs {roll.total_time(angle):~.1f}, "
          f"{margin.to('percent'):~+.1f} -- {verdict}")
    print(f"{'':>17}largest slew that would fit: "
          f"{roll.achievable_angle(target):~.1f}")
    print(f"{'':>17}ground track runs {roll.ground_distance(target):~.0f} meanwhile")
    print()

40 deg in 113 s:  needs 83.2 s, +35.8 % -- fits
                 largest slew that would fit: 62.0 deg
                 ground track runs 798 km meanwhile

40 deg in 90 s:  needs 83.2 s, +8.2 % -- fits
                 largest slew that would fit: 45.0 deg
                 ground track runs 635 km meanwhile



## Which spacecraft are we slewing?

The config names three inertia cases. `first_guess` estimates the inertia from a box and takes its mass from the mass budget, so it moves whenever the equipment list does. `measured_bol` and `measured_eol` state an inertia outright, as a mass properties report gives it, and need no mass at all.

The names are ours — nothing in the code matches on them. What matters is that a number can always be traced to the spacecraft that produced it.

In [17]:
cases = []
for name in config.inertia_cases:
    case = AgilityBudget(
        config, orbit, Axis.ROLL, name, mass_budget=mass_data
    )
    cases.append({
        "case": name,
        "shape": "envelope" if case.case.is_envelope else "stated",
        "inertia": case.inertia.magnitude,
        "max_rate": case.max_rate().magnitude,
        "slew_40_deg": case.slew_time(40 * u.deg).magnitude,
        "achievable_in_113_s": case.achievable_angle(113 * u.s).magnitude,
    })

pd.DataFrame(cases).style.format(
    {"inertia": "{:,.1f}", "max_rate": "{:,.4f}",
     "slew_40_deg": "{:,.2f}", "achievable_in_113_s": "{:,.2f}"}
).hide(  # pyright: ignore[reportAttributeAccessIssue]
    axis="index"
)

case,shape,inertia,max_rate,slew_40_deg,achievable_in_113_s
first_guess,envelope,262.3,0.7366,63.19,61.96
measured_bol,stated,264.7,0.7298,63.69,61.39
measured_eol,stated,250.0,0.7727,60.65,65.00


The spread is worth reading. `measured_eol` is the same spacecraft with its tanks empty: lighter, so it turns faster and reaches further inside the same allocation. Agility is one of the few budgets that *improves* over a mission.

## The whole thing, in one table

`tabulated_agility()` lays the slew table out as a document. Given a target duration it also checks each angle against it and marks what does not fit; without one those columns are left out entirely, because there is nothing to check against.

In [18]:
print(f"inertia case  '{roll.case_name}'  {roll.inertia:~.1f}")
print()
roll.tabulated_agility(target_duration=113 * u.s)

inertia case  'first_guess'  262.3 kg * m ** 2



Slew [deg],Limited by,Slew [s],With settling [s],Peak rate [deg/s],Momentum used,Time margin,
5,torque,15.5,35.5,0.6440,87.4%,+218.1%,PASS
10,momentum,22.5,42.5,0.7366,100.0%,+166.2%,PASS
15,momentum,29.2,49.2,0.7366,100.0%,+129.5%,PASS
20,momentum,36.0,56.0,0.7366,100.0%,+101.7%,PASS
40,momentum,63.2,83.2,0.7366,100.0%,+35.8%,PASS
45,momentum,70.0,90.0,0.7366,100.0%,+25.6%,PASS
60,momentum,90.3,110.3,0.7366,100.0%,+2.4%,PASS
90,momentum,131.1,151.1,0.7366,100.0%,-25.2%,FAILS


## One wheel failed

With one of four gone, only two of the three survivors can be driven at full torque if the net in-plane momentum is to stay zero. Both the momentum and the torque about the axis halve, so the rate halves and the acceleration halves with it.

This is the same pyramid flying degraded, not a three-wheel mounting: the geometry and the projection are unchanged, only the usable count drops.

In [19]:
nominal = roll.slew_table()[["angle", "slew_time"]]
degraded = roll.slew_table(degraded=True)[["angle", "slew_time"]]

comparison = nominal.merge(degraded, on="angle", suffixes=("_4_wheels", "_3_wheels"))
comparison["penalty"] = (
    comparison["slew_time_3_wheels"] / comparison["slew_time_4_wheels"] - 1
)
comparison.style.format(
    {"angle": "{:,.0f}", "slew_time_4_wheels": "{:,.1f}",
     "slew_time_3_wheels": "{:,.1f}", "penalty": "{:+.0%}"}
).hide(  # pyright: ignore[reportAttributeAccessIssue]
    axis="index"
)

angle,slew_time_4_wheels,slew_time_3_wheels,penalty
5,15.5,22.5,+45%
10,22.5,36.0,+60%
15,29.2,49.6,+70%
20,36.0,63.2,+75%
40,63.2,117.5,+86%
45,70.0,131.1,+87%
60,90.3,171.8,+90%
90,131.1,253.3,+93%
